# Lab 13 — Eigenvectors: Hidden Directions That Survive Transformation

In Chapter 13, we learned that a matrix does not treat every direction equally.

Most vectors turn when a matrix acts on them. Eigenvectors are the special directions that do **not** turn. They may stretch, shrink, flip, or collapse, but they stay on the same line.

This lab is more than a quick calculation practice. We will:

- visualize matrix action on many directions,
- discover eigenvectors geometrically,
- compute eigenvalues and eigenvectors with Python,
- study repeated matrix action,
- explore ranking and stability,
- connect eigenvectors to data and PCA intuition,
- end with high-dimensional eigenvector behavior.

The central equation is

$$
Av = \lambda v.
$$

## 0. Setup

Run this cell first. We use only standard scientific Python packages.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. A helper function for drawing vectors

We will repeatedly draw input vectors and output vectors. The following function keeps the visuals consistent.

In [ ]:
def draw_vector(ax, v, label=None, linestyle='-', linewidth=2):
    v = np.asarray(v, dtype=float)
    ax.arrow(0, 0, v[0], v[1],
             head_width=0.06, length_includes_head=True,
             linewidth=linewidth, linestyle=linestyle)
    if label is not None:
        ax.text(v[0]*1.08, v[1]*1.08, label, fontsize=12)

def setup_plane(ax, lim=4, title=None):
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    if title:
        ax.set_title(title)

## 2. Most vectors turn

Let

$$
A=\begin{bmatrix}2 & 1\\0 & 1\end{bmatrix}.
$$

We compare several input vectors $x$ with their outputs $Ax$.

In [ ]:
A = np.array([[2, 1],
              [0, 1]], dtype=float)

vectors = [np.array([1, 0]), np.array([0, 1]), np.array([1, 1]), np.array([-1, 1])]

fig, ax = plt.subplots(figsize=(7, 7))
setup_plane(ax, lim=4, title='Input vectors and their outputs under A')

for i, x in enumerate(vectors):
    y = A @ x
    draw_vector(ax, x, label=f'x{i+1}', linestyle='--')
    draw_vector(ax, y, label=f'Ax{i+1}')

plt.show()

### Reflection

Which input vector appears to stay on the same line after applying $A$? Which vectors clearly turn?

## 3. Checking the eigenvector equation directly

A vector $v$ is an eigenvector if $Av$ is a scalar multiple of $v$.

We can check this by comparing $Av$ with $\lambda v$.

In [ ]:
v = np.array([1, 0], dtype=float)
Av = A @ v

print('v  =', v)
print('Av =', Av)
print('Is Av equal to 2v?', np.allclose(Av, 2*v))

Now try a vector that is not an eigenvector.

In [ ]:
x = np.array([1, 1], dtype=float)
Ax = A @ x

print('x  =', x)
print('Ax =', Ax)
print('The direction changed: Ax is not a scalar multiple of x.')

## 4. Computing eigenvalues and eigenvectors with NumPy

`np.linalg.eig(A)` returns two objects:

1. an array of eigenvalues,
2. a matrix whose columns are eigenvectors.

In [ ]:
values, eigvecs = np.linalg.eig(A)

print('Eigenvalues:')
print(values)
print('\nEigenvectors stored as columns:')
print(eigvecs)

for j in range(len(values)):
    lam = values[j]
    v = eigvecs[:, j]
    print('\nPair', j+1)
    print('lambda =', lam)
    print('v =', v)
    print('A v =', A @ v)
    print('lambda v =', lam * v)

### Student task

Change the matrix $A$ above to

$$
\begin{bmatrix}3 & 1\\0 & 2\end{bmatrix}
$$

and rerun the eigenvalue computation. Which direction has the larger stretch factor?

## 5. Visual discovery: transform the unit circle

A powerful picture is to transform the unit circle under a matrix. For a symmetric matrix, the circle becomes an ellipse. The eigenvectors point along the ellipse axes.

In [ ]:
B = np.array([[2, 1],
              [1, 2]], dtype=float)

values, eigvecs = np.linalg.eig(B)

theta = np.linspace(0, 2*np.pi, 500)
circle = np.vstack([np.cos(theta), np.sin(theta)])
image = B @ circle

fig, ax = plt.subplots(figsize=(7, 7))
setup_plane(ax, lim=4, title='Unit circle transformed by a symmetric matrix')
ax.plot(circle[0], circle[1], label='unit circle')
ax.plot(image[0], image[1], label='transformed circle')

for j in range(2):
    v = eigvecs[:, j]
    v = v / np.linalg.norm(v)
    draw_vector(ax, 1.5*v, label=f'eig {j+1}')
    draw_vector(ax, -1.5*v)

ax.legend()
plt.show()

print('Eigenvalues:', values)
print('Eigenvectors:')
print(eigvecs)

## 6. Positive, negative, and zero eigenvalues

Eigenvalues tell us what happens along special directions.

Let us compare three simple matrices.

In [ ]:
matrices = {
    'stretch': np.array([[3, 0], [0, 1]], dtype=float),
    'flip and stretch': np.array([[-2, 0], [0, 1]], dtype=float),
    'projection': np.array([[1, 0], [0, 0]], dtype=float)
}

for name, M in matrices.items():
    vals, vecs = np.linalg.eig(M)
    print('\n', name.upper())
    print(M)
    print('eigenvalues:', vals)

### Interpretation task

For each matrix above, write one sentence explaining what happens to the eigenvector directions.

## 7. A shear: one eigen-direction is not enough to explain everything

A shear matrix preserves one direction but slides all other directions.

$$
S=\begin{bmatrix}1&2\\0&1\end{bmatrix}.
$$

In [ ]:
S = np.array([[1, 2],
              [0, 1]], dtype=float)
vals, vecs = np.linalg.eig(S)

print('Eigenvalues:', vals)
print('Eigenvectors:')
print(vecs)

# Draw a grid before and after shearing
xs = np.linspace(-2, 2, 9)
fig, ax = plt.subplots(figsize=(7, 7))
setup_plane(ax, lim=5, title='A shear preserves one direction and slides others')

for c in xs:
    line1 = np.vstack([np.linspace(-2, 2, 100), np.full(100, c)])
    line2 = np.vstack([np.full(100, c), np.linspace(-2, 2, 100)])
    img1 = S @ line1
    img2 = S @ line2
    ax.plot(img1[0], img1[1], linewidth=0.8)
    ax.plot(img2[0], img2[1], linewidth=0.8)

draw_vector(ax, [2, 0], label='eigen-direction')
plt.show()

## 8. Repeated matrix action

If $Av=\lambda v$, then

$$
A^k v = \lambda^k v.
$$

This is why eigenvalues control long-term behavior.

In [ ]:
C = np.array([[1.2, 0.3],
              [0.1, 0.8]], dtype=float)

vals, vecs = np.linalg.eig(C)
print('Eigenvalues:', vals)

x = np.array([2.0, 1.0])
trajectory = [x]
for k in range(20):
    x = C @ x
    trajectory.append(x)
trajectory = np.array(trajectory)

fig, ax = plt.subplots(figsize=(7, 7))
setup_plane(ax, lim=max(4, np.max(np.abs(trajectory))*1.1), title='Repeated matrix action')
ax.plot(trajectory[:,0], trajectory[:,1], marker='o')
for k in [0, 1, 2, 5, 10, 20]:
    ax.text(trajectory[k,0], trajectory[k,1], str(k))

# dominant eigenvector
j = np.argmax(np.abs(vals))
v = vecs[:, j].real
v = v / np.linalg.norm(v)
draw_vector(ax, 3*v, label='dominant eigendir')
draw_vector(ax, -3*v)
plt.show()

### What do you observe?

Does the trajectory begin to align with the dominant eigenvector? Why should this happen?

## 9. Ranking as an eigenvector problem

Suppose four pages link to each other. A page is important if important pages point to it. This creates a self-consistency equation.

Below, we build a simple link matrix and repeatedly apply it to a ranking vector.

In [ ]:
# Columns represent where each page distributes its vote.
P = np.array([
    [0.0, 1/2, 1/2, 0.0],
    [1/3, 0.0, 0.0, 1/2],
    [1/3, 1/2, 0.0, 1/2],
    [1/3, 0.0, 1/2, 0.0]
])

# Verify columns sum to 1
print('Column sums:', P.sum(axis=0))

r = np.ones(4) / 4
history = [r]
for k in range(30):
    r = P @ r
    history.append(r)
history = np.array(history)

print('Final ranking:', r)

plt.figure(figsize=(8, 5))
for i in range(4):
    plt.plot(history[:, i], marker='o', label=f'Page {i+1}')
plt.xlabel('iteration')
plt.ylabel('rank score')
plt.title('Ranking by repeated matrix updates')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

vals, vecs = np.linalg.eig(P)
idx = np.argmin(np.abs(vals - 1))
v = vecs[:, idx].real
v = v / v.sum()
print('Eigenvector ranking for eigenvalue 1:', v)

## 10. Data variation and eigenvectors

In data analysis, eigenvectors of a covariance matrix show directions of variation.

We create a 2D dataset that is stretched along a diagonal direction.

In [ ]:
rng = np.random.default_rng(7)

# Create correlated data by transforming random normal data
Z = rng.normal(size=(400, 2))
T = np.array([[3.0, 1.4],
              [1.0, 0.8]])
X = Z @ T.T
X_centered = X - X.mean(axis=0)

Cov = np.cov(X_centered, rowvar=False)
vals, vecs = np.linalg.eig(Cov)
order = np.argsort(vals)[::-1]
vals = vals[order]
vecs = vecs[:, order]

fig, ax = plt.subplots(figsize=(7, 7))
setup_plane(ax, lim=8, title='Eigenvectors of covariance describe data variation')
ax.scatter(X_centered[:,0], X_centered[:,1], s=12, alpha=0.4)

for j in range(2):
    v = vecs[:, j]
    scale = np.sqrt(vals[j])
    draw_vector(ax, scale*v, label=f'PC {j+1}')
    draw_vector(ax, -scale*v)

plt.show()

print('Covariance matrix:')
print(Cov)
print('Eigenvalues:', vals)
print('Eigenvectors:')
print(vecs)

The first eigenvector points in the direction of largest variation. This idea becomes PCA later in the book.

## 11. Image experiment: eigenvectors of a blur-like matrix

A smoothing matrix averages neighboring pixels. Its eigenvectors are special patterns that keep their shape under smoothing. Low-frequency patterns survive; high-frequency patterns shrink.

In [ ]:
def smoothing_matrix(n):
    M = np.zeros((n, n))
    for i in range(n):
        M[i, i] = 0.5
        if i > 0:
            M[i, i-1] = 0.25
        if i < n-1:
            M[i, i+1] = 0.25
    return M

n = 40
M = smoothing_matrix(n)
vals, vecs = np.linalg.eig(M)
idx = np.argsort(vals)[::-1]
vals = vals[idx].real
vecs = vecs[:, idx].real

plt.figure(figsize=(9, 5))
for j in [0, 1, 2, 8, 20, 35]:
    plt.plot(vecs[:, j], label=f'eigenvalue {vals[j]:.3f}')
plt.title('Eigenvectors of a smoothing matrix')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Interpretation

The eigenvectors with large eigenvalues are smooth patterns. The eigenvectors with small eigenvalues oscillate quickly and are strongly damped by smoothing.

## 12. High-dimensional experiment: dominant eigenvectors

Large real datasets may have hundreds or thousands of dimensions. Eigenvectors still help identify dominant directions.

We create high-dimensional data with one strong hidden direction and see if the top eigenvector can find it.

In [ ]:
rng = np.random.default_rng(42)
n_samples = 800
n_features = 80

# Hidden direction
u = rng.normal(size=n_features)
u = u / np.linalg.norm(u)

# Data = strong signal along u + noise
scores = rng.normal(scale=4.0, size=n_samples)
noise = rng.normal(scale=1.0, size=(n_samples, n_features))
X = scores[:, None] * u[None, :] + noise
X = X - X.mean(axis=0)

Cov = (X.T @ X) / (n_samples - 1)
vals, vecs = np.linalg.eigh(Cov)  # eigh is best for symmetric matrices
idx = np.argsort(vals)[::-1]
vals = vals[idx]
vecs = vecs[:, idx]

top = vecs[:, 0]
cosine = abs(np.dot(top, u) / (np.linalg.norm(top) * np.linalg.norm(u)))

print('Top 10 eigenvalues:')
print(vals[:10])
print('\nCosine similarity between true hidden direction and top eigenvector:')
print(cosine)

plt.figure(figsize=(8, 4))
plt.plot(vals[:30], marker='o')
plt.xlabel('index')
plt.ylabel('eigenvalue')
plt.title('Spectrum of high-dimensional covariance matrix')
plt.grid(True, alpha=0.3)
plt.show()

### Final reflection

The top eigenvector recovered the hidden direction because the data had much larger variation along that direction than in random noise directions.

This is the seed of PCA.

## 13. Student extensions

Choose one or more:

1. Change the strength of the hidden signal in the high-dimensional experiment. When does the top eigenvector fail?
2. Create a matrix with eigenvalues $3$, $1$, and $0.2$. Study repeated matrix action.
3. Build your own small ranking matrix and compute its dominant eigenvector.
4. Create a matrix that rotates and stretches. What happens to its eigenvalues?
5. Use the smoothing matrix on a noisy signal and explain the role of its eigenvectors.

## Key takeaway

Eigenvectors are hidden directions of a matrix. Eigenvalues tell how the matrix behaves along those directions. When a matrix is repeated, when a system evolves, or when data has hidden structure, eigenvectors often reveal the main story.